# TradeMetrics — Validating the Engine Against Known Truth

**What this notebook is.** Every number here is computed from *simulated* data whose
true parameters I chose in advance. That is the whole point: if I simulate a return
series with a true annualised Sharpe of exactly 1.0, I can check whether the engine's
Sharpe estimator returns something close to 1.0, and whether its confidence interval
covers the truth as often as it claims to.

**What this notebook is not.** It contains no real trading data. My personal brokerage
account, its NAV, and its trade history are private and deliberately absent from this
repository. Nothing here is evidence about my own returns, and no claim in this
repository depends on data a reader cannot regenerate.

**Why this framing.** A portfolio-analytics repo that says "here is my Sharpe ratio"
asks you to trust an unverifiable number. A repo that says "here is the estimator, and
here is the evidence that it is unbiased and that its error bars are honest" makes a
claim you can check line by line. The second is the more defensible artifact, and it is
the one that survives contact with someone who knows statistics.

**Sections**

1. Estimator correctness — does each metric recover a known truth?
2. Uncertainty calibration — are the Sharpe confidence intervals honest?
3. Skill vs luck — how much track record does it take to prove anything?
4. The beta illusion — why leveraged beta looks like skill in a bull market
5. FIFO engine validation — hand-computed trade sequences
6. Limitations, written as the skeptical reviewer
7. Conclusion — what this engine can and cannot claim

Run it end to end with `jupyter nbconvert --to notebook --execute`. It is deterministic:
every random draw is seeded.

In [ ]:
import sys
from pathlib import Path

# Allow the notebook to import the package whether it is run from the repo root
# (as CI does) or from analysis/ (as a human double-clicking it would).
repo_root = Path.cwd()
if not (repo_root / "trademetrics").is_dir():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from trademetrics.benchmark import BenchmarkAnalysis
from trademetrics.fifo import FifoEngine
from trademetrics.metrics import MetricsEngine, sharpe_with_ci
from trademetrics.simulate import simulate_returns
from trademetrics.sizing import erc_weights

TRADING_DAYS = 252
plt.rcParams.update({
    "figure.figsize": (10, 4.2), "axes.grid": True, "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 10,
})

print(f"repo root: {repo_root}")
print(f"pandas {pd.__version__} · numpy {np.__version__}")

---
## 1. Estimator correctness

The claim to establish: each metric returns the population value it is supposed to
estimate. I test this two ways — by simulating large samples from a distribution with
known parameters and checking convergence, and by computing one worked example per
metric by hand and comparing it to the engine.

### 1.1 Sharpe ratio

`simulate_returns(n, true_sharpe, vol_ann)` draws IID normal daily returns whose
population annualised Sharpe is exactly `true_sharpe`. With a very long sample the
estimate should sit essentially on top of the truth.

In [ ]:
TRUE_SHARPE = 1.0
VOL = 0.15
N_REPS = 400

rows = []
for n in [252, 252 * 5, 252 * 20, 252 * 100]:
    # Many INDEPENDENT track records of length n. Averaging over them shows
    # whether the estimator is unbiased; their spread is the true sampling
    # error, which we compare against what the Lo formula predicts.
    ests = [
        sharpe_with_ci(
            simulate_returns(n=n, true_sharpe=TRUE_SHARPE, vol_ann=VOL, seed=90000 + k),
            rf=0.0,
        )
        for k in range(N_REPS)
    ]
    point = np.array([e.sharpe for e in ests])
    rows.append({
        "years": n / TRADING_DAYS,
        "n_days": n,
        "mean estimate": point.mean(),
        "bias": point.mean() - TRUE_SHARPE,
        "empirical SD": point.std(ddof=1),
        "mean Lo SE": np.mean([e.se for e in ests]),
    })

convergence = pd.DataFrame(rows)
print(f"True annualised Sharpe = {TRUE_SHARPE}, averaged over {N_REPS} "
      f"independent track records per row\n")
print(convergence.round(3).to_string(index=False))

# A single realisation, to show what one track record actually looks like.
one = sharpe_with_ci(
    simulate_returns(n=252, true_sharpe=TRUE_SHARPE, vol_ann=VOL, seed=12345), rf=0.0
)
print(f"\nOne arbitrary single year (seed 12345): estimate {one.sharpe:.3f}, "
      f"95% CI {one.ci_low:.2f} to {one.ci_high:.2f}")

Two things to read off this table. The **bias column is near zero at every sample
length**, which is what "the estimator is correct" means — averaged over many track
records it lands on the truth rather than systematically above or below it. And the
**empirical SD matches the mean Lo SE** closely at every length, which says the
analytical standard-error formula is describing the real sampling variation rather
than an optimistic approximation of it.

The error bar shrinks roughly as $1/\sqrt{n}$: quadrupling the sample halves it.

The single realisation printed underneath is the important caveat. *One* year, drawn
from a world where the true Sharpe is exactly 1.0, produced an estimate nowhere near
1.0 with an interval about four Sharpe units wide. Unbiased does not mean accurate on
one sample. Hold that thought for sections 2 and 3.

**Worked example by hand.** Six returns, computed with a calculator rather than the
engine, then compared.

In [ ]:
r_hand = np.array([0.01, -0.005, 0.002, 0.007, -0.001, 0.004])

# By hand, from the definitions:
mean_hand = r_hand.sum() / len(r_hand)
var_hand = ((r_hand - mean_hand) ** 2).sum() / (len(r_hand) - 1)   # ddof = 1
sd_hand = var_hand ** 0.5
sharpe_daily_hand = mean_hand / sd_hand
sharpe_ann_hand = sharpe_daily_hand * np.sqrt(252)

# Lo (2002) IID standard error: SE(SR_daily) = sqrt((1 + SR_daily^2 / 2) / n)
se_daily_hand = ((1 + sharpe_daily_hand ** 2 / 2) / len(r_hand)) ** 0.5
se_ann_hand = se_daily_hand * np.sqrt(252)

engine_est = sharpe_with_ci(r_hand, rf=0.0)

print(f"mean daily          hand {mean_hand:.10f}")
print(f"sd daily (ddof=1)   hand {sd_hand:.10f}")
print(f"Sharpe annualised   hand {sharpe_ann_hand:.10f}   engine {engine_est.sharpe:.10f}")
print(f"Lo SE annualised    hand {se_ann_hand:.10f}   engine {engine_est.se:.10f}")
assert abs(sharpe_ann_hand - engine_est.sharpe) < 1e-12
assert abs(se_ann_hand - engine_est.se) < 1e-12
print("\nhand calculation and engine agree to machine precision")

### 1.2 Sortino ratio

The bug this replaces is worth stating precisely, because it is the single most common
error in retail performance code. Downside deviation is

$$\sigma_d = \sqrt{\frac{1}{N}\sum_{t=1}^{N} \min(r_t - \tau, 0)^2}$$

The sum runs over **all $N$ observations**, not only the ones below target. Days above
target contribute zero to the numerator but still count in $N$. The common mistake —
taking the standard deviation of the negative-return subset — divides by a smaller
denominator *and* re-centres on the mean of the losses, which inflates Sortino
substantially. Here is the size of that error.

In [ ]:
r_s = np.array([0.02, -0.01, 0.01, -0.03])
target = 0.0

# Correct: mean of squared shortfalls over ALL observations
shortfall = np.minimum(r_s - target, 0.0)
downside_correct = np.sqrt(np.mean(shortfall ** 2))

# The common bug: std of just the negative returns
downside_buggy = r_s[r_s < 0].std(ddof=1)

excess_ann = r_s.mean() * TRADING_DAYS
sortino_correct = excess_ann / (downside_correct * np.sqrt(TRADING_DAYS))
sortino_buggy = excess_ann / (downside_buggy * np.sqrt(TRADING_DAYS))

dates = pd.bdate_range("2024-01-01", periods=len(r_s) + 1)
nav_s = pd.Series(100.0 * np.cumprod(np.r_[1.0, 1 + r_s]), index=dates)
engine_sortino = MetricsEngine(nav_s, rf=0.0).sortino_ratio()

print(f"downside deviation, correct (all obs)   {downside_correct:.10f}")
print(f"downside deviation, buggy  (neg subset) {downside_buggy:.10f}")
print(f"Sortino, correct  {sortino_correct:.6f}   engine {engine_sortino:.6f}")
print(f"Sortino, buggy    {sortino_buggy:.6f}")
print(f"\nthe buggy version overstates the ratio by "
      f"{abs(sortino_buggy / sortino_correct) - 1:.1%} on this sample")
assert abs(sortino_correct - engine_sortino) < 1e-12

On this four-observation sample the buggy formula flatters the ratio by about 12%. On
a real return series with many small positive days and few large negative ones — the
typical shape — the distortion is larger, because dropping the positive days removes
most of the sample from the denominator's count.

**Target choice.** I use $\tau = r_f/252$, the daily risk-free rate, so that Sortino and
Sharpe answer the same question ("return per unit of risk, above the risk-free
alternative") and differ only in how they define risk. Setting $\tau = 0$ is also
defensible and is what most retail tools do; it produces a higher number, and mixing the
two conventions across a comparison is how people accidentally cherry-pick.

### 1.3 Value at Risk

Two independent estimators of the same quantity, which should agree when the data really
is normal and diverge when it is not. That divergence is the diagnostic.

In [ ]:
rng = np.random.default_rng(2024)
n_var = 3000

normal_returns = rng.normal(0.0004, 0.010, n_var)
# Fat-tailed alternative: same mean and variance, Student-t shaped
t_raw = rng.standard_t(df=3, size=n_var)
t_returns = 0.0004 + 0.010 * t_raw / np.sqrt(3 / (3 - 2))  # rescale to unit variance

def var_pair(returns, label):
    dates = pd.bdate_range("2000-01-03", periods=len(returns) + 1)
    nav = pd.Series(100.0 * np.cumprod(np.r_[1.0, 1 + returns]), index=dates)
    eng = MetricsEngine(nav, rf=0.0)
    return {
        "series": label,
        "excess kurtosis": float(pd.Series(returns).kurt()),
        "VaR95 parametric": eng.var_parametric(0.95),
        "VaR95 historical": eng.var_historical(0.95),
        "VaR99 parametric": eng.var_parametric(0.99),
        "VaR99 historical": eng.var_historical(0.99),
    }

var_table = pd.DataFrame([
    var_pair(normal_returns, "normal"),
    var_pair(t_returns, "fat-tailed (t, df=3)"),
])
print(var_table.round(4).to_string(index=False))

On normally distributed returns the two VaR estimates agree closely — as they must,
since the parametric formula assumes exactly the distribution the data was drawn from.

On fat-tailed returns with the *same* mean and variance, they come apart in a specific
and diagnostic way. At the 95% level the parametric figure is the **larger** of the two;
at 99% it is the **smaller**, understating the loss the empirical quantile actually
shows. That crossover is the signature of excess kurtosis: a fat-tailed distribution has
more mass in the extreme tail and more mass near the centre, with less in between. The
normal approximation therefore overstates moderate losses and understates severe ones —
and it is severe losses that a risk limit exists to control.

This is why the engine reports both and why the dashboard shows both. A single
parametric VaR on real equity returns is a number that is wrong in a predictable
direction, and reporting it alone invites exactly the question you do not want to be
surprised by in an interview.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for ax, (data, label) in zip(
    axes, [(normal_returns, "Normal returns"), (t_returns, "Fat-tailed returns (t, df=3)")]
):
    dates = pd.bdate_range("2000-01-03", periods=len(data) + 1)
    nav = pd.Series(100.0 * np.cumprod(np.r_[1.0, 1 + data]), index=dates)
    eng = MetricsEngine(nav, rf=0.0)
    ax.hist(data, bins=120, color="#4a6080", alpha=0.75)
    ax.axvline(-eng.var_parametric(0.99), color="#0099ff", lw=2,
               label=f"99% parametric {eng.var_parametric(0.99):.3f}")
    ax.axvline(-eng.var_historical(0.99), color="#ff4757", lw=2, ls="--",
               label=f"99% historical {eng.var_historical(0.99):.3f}")
    ax.set_title(label)
    ax.set_xlim(-0.06, 0.06)
    ax.legend(fontsize=8, frameon=False)
axes[0].set_ylabel("days")
fig.suptitle("Parametric VaR agrees with historical only when returns are normal", y=1.02)
plt.tight_layout()
plt.show()

### 1.4 Maximum drawdown and CAPM beta

Two more hand-checks. Drawdown is deterministic given a NAV path, so the check is exact.
Beta is checked against a construction where the answer is known with certainty: a
portfolio that is a noiseless leveraged clone of the benchmark must have beta equal to
its leverage and alpha of exactly zero.

In [ ]:
# Max drawdown — hand traceable
nav_dd = pd.Series(
    [100.0, 110.0, 99.0, 105.0, 120.0, 90.0],
    index=pd.bdate_range("2024-01-01", periods=6),
)
# running peaks: 100, 110, 110, 110, 120, 120
# drawdowns:       0%,   0%, -10%, -4.5%,  0%, -25%   -> worst is -25%
print(f"max drawdown  hand -0.25   engine {MetricsEngine(nav_dd).max_drawdown():.10f}")

# CAPM on a noiseless 1.5x leveraged clone of the benchmark
rng_b = np.random.default_rng(99)
n_b = 500
rf_ann = 0.02
rf_d = rf_ann / TRADING_DAYS
bench_excess = rng_b.normal(0.0003, 0.008, n_b)
dates_b = pd.bdate_range("2023-01-02", periods=n_b + 1)
bench_prices = pd.DataFrame(
    {"SPY": 100.0 * np.cumprod(np.r_[1.0, 1 + rf_d + bench_excess])}, index=dates_b
)
port_returns = pd.Series(rf_d + 1.5 * bench_excess, index=dates_b[1:])

est = BenchmarkAnalysis(port_returns, bench_prices, rf=rf_ann).capm("SPY")
print(f"\nnoiseless 1.5x clone:")
print(f"  beta      true 1.5    estimated {est.beta:.10f}")
print(f"  alpha     true 0.0    estimated {est.alpha_ann:.10f}")
print(f"  R-squared true 1.0    estimated {est.r_squared:.10f}")

---
## 2. Uncertainty calibration — are the error bars honest?

A point estimate with a confidence interval makes a testable promise: *if I repeat this
experiment many times, the interval will contain the true value about 95% of the time.*
An interval that fails this test is worse than no interval, because it dresses up a
guess as a measurement.

So I test it. Simulate 2,000 independent one-year track records from a world where the
true Sharpe is exactly 1.0, compute the Lo (2002) interval for each, and count how often
it covers 1.0.

In [ ]:
N_TRIALS = 2000
TRUE_SR = 1.0

def coverage_experiment(n_days, true_sr=TRUE_SR, n_trials=N_TRIALS, seed0=1000):
    estimates, covered = [], 0
    for i in range(n_trials):
        r = simulate_returns(n=n_days, true_sharpe=true_sr, vol_ann=0.15, seed=seed0 + i)
        est = sharpe_with_ci(r, rf=0.0)
        estimates.append(est.sharpe)
        if est.ci_low <= true_sr <= est.ci_high:
            covered += 1
    return np.array(estimates), covered / n_trials

est_1y, cov_1y = coverage_experiment(252)
est_5y, cov_5y = coverage_experiment(252 * 5)

print(f"True Sharpe = {TRUE_SR}, {N_TRIALS} simulated track records\n")
print(f"  1 year   coverage of nominal 95% CI : {cov_1y:.1%}")
print(f"  5 years  coverage of nominal 95% CI : {cov_5y:.1%}")
print(f"\n  1 year   mean estimate {est_1y.mean():.3f}, "
      f"empirical SD of estimates {est_1y.std(ddof=1):.3f}")
print(f"  5 years  mean estimate {est_5y.mean():.3f}, "
      f"empirical SD of estimates {est_5y.std(ddof=1):.3f}")

Coverage comes out close to the nominal 95%, so the interval is doing what it claims.

The second pair of numbers is the one that matters for interpreting any real track
record. **The empirical standard deviation of a one-year Sharpe estimate is close to
1.0.** Not 0.1 — 1.0, the same magnitude as the quantity being estimated. A trader whose
true Sharpe is 1.0 will, over a single year, routinely report anything from roughly
−1 to +3 purely through sampling variation.

Let me also confirm the analytical Lo standard error matches the empirical spread, since
that is what licenses using the formula on a single sample where the spread cannot be
observed.

In [ ]:
r_one = simulate_returns(n=252, true_sharpe=TRUE_SR, vol_ann=0.15, seed=777)
analytic_se = sharpe_with_ci(r_one, rf=0.0).se

# Non-parametric alternative: bootstrap the same single sample
BOOT = 4000
boot_rng = np.random.default_rng(31337)
boot = np.array([
    sharpe_with_ci(boot_rng.choice(r_one, size=len(r_one), replace=True), rf=0.0).sharpe
    for _ in range(BOOT)
])

point = sharpe_with_ci(r_one, rf=0.0).sharpe

print(f"Standard error of a 1-year Sharpe estimate, three ways:\n")
print(f"  Lo (2002) analytic, from one sample : {analytic_se:.3f}")
print(f"  bootstrap SD, from the same sample  : {boot.std(ddof=1):.3f}")
print(f"  empirical SD across {N_TRIALS} simulated years : {est_1y.std(ddof=1):.3f}")
print(f"\nThis particular simulated year (true Sharpe {TRUE_SR}):")
print(f"  point estimate         {point:+.3f}")
print(f"  bootstrap 95% interval [{np.percentile(boot, 2.5):.2f}, "
      f"{np.percentile(boot, 97.5):.2f}]")
print(f"  interval contains the true {TRUE_SR}? "
      f"{np.percentile(boot, 2.5) <= TRUE_SR <= np.percentile(boot, 97.5)}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.8))
ax.hist(est_1y, bins=60, alpha=0.75, color="#0099ff", label="1-year estimates")
ax.hist(est_5y, bins=60, alpha=0.7, color="#00d4aa", label="5-year estimates")
ax.axvline(TRUE_SR, color="#ff4757", lw=2, label=f"true Sharpe = {TRUE_SR}")
ax.axvline(0, color="#8fa3c0", lw=1, ls=":", label="zero skill")
ax.set_xlabel("estimated annualised Sharpe")
ax.set_ylabel("simulated track records")
ax.set_title("The same trader, measured over 1 year vs 5 years")
ax.legend(frameon=False, fontsize=9)
plt.tight_layout()
plt.show()

frac_below_zero = (est_1y < 0).mean()
frac_above_two = (est_1y > 2).mean()
print(f"Of {N_TRIALS} one-year records generated by a genuinely skilled trader "
      f"(true Sharpe 1.0):")
print(f"  {frac_below_zero:.1%} reported a NEGATIVE Sharpe")
print(f"  {frac_above_two:.1%} reported a Sharpe above 2.0")

---
## 3. Skill versus luck

Section 2 says a one-year Sharpe is noisy. This section turns that into the two
questions an interviewer actually asks: *how long before the number means anything*,
and *what happens when you pick the best performer out of many?*

### 3.1 How much track record is needed?

To reject "true Sharpe is zero" at 95% confidence I need the estimate to exceed roughly
$1.96 \times SE$. Under the Lo formula with true Sharpe $S$ over $y$ years,
$SE \approx \sqrt{(1 + S^2/2)/(252y)} \times \sqrt{252} = \sqrt{(1 + S^2/2)/y}$.
Setting $S = 1.96\,SE$ and solving for $y$ gives the required track-record length.

In [ ]:
def years_to_significance(true_sharpe, z=1.959963984540054):
    # Years of daily data needed for the expected estimate to exceed z * SE.
    return z ** 2 * (1 + true_sharpe ** 2 / 2) / true_sharpe ** 2

table = pd.DataFrame({
    "true Sharpe": [0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0],
})
table["years needed"] = table["true Sharpe"].map(years_to_significance)
table["trading days"] = (table["years needed"] * TRADING_DAYS).round(0).astype(int)
print("Track record required to distinguish a given true Sharpe from zero at 95%:\n")
print(table.round(2).to_string(index=False))

A genuinely good manager running at a true Sharpe of 1.0 needs roughly **six years** of
daily data before a single-sample test can distinguish them from someone with no skill
at all. At a true Sharpe of 0.5 it is over thirty years. This is not a quirk of the
formula; it is the arithmetic of measuring a small mean against a large variance.

The practical consequence: any one-year Sharpe — mine, a fund's, anyone's — is
compatible with a very wide range of underlying skill, and quoting it to two decimal
places without an interval implies a precision that does not exist.

### 3.2 The selection effect

The situation gets worse when the reported number was chosen for being good. Simulate
1,000 traders who all have *exactly zero* skill, let each run for a year, and look at
the best one.

In [ ]:
N_TRADERS = 1000
sel_rng = np.random.default_rng(4242)

zero_skill = np.array([
    sharpe_with_ci(
        sel_rng.normal(0.0, 0.15 / np.sqrt(TRADING_DAYS), TRADING_DAYS), rf=0.0
    ).sharpe
    for _ in range(N_TRADERS)
])

print(f"{N_TRADERS} traders, all with TRUE Sharpe = 0, one year each:\n")
print(f"  mean reported Sharpe      {zero_skill.mean():+.3f}")
print(f"  best reported Sharpe      {zero_skill.max():+.3f}")
print(f"  90th percentile           {np.percentile(zero_skill, 90):+.3f}")
print(f"  99th percentile           {np.percentile(zero_skill, 99):+.3f}")
print(f"  fraction reporting > 1.0  {(zero_skill > 1.0).mean():.1%}")
print(f"  fraction reporting > 1.5  {(zero_skill > 1.5).mean():.1%}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.8))
ax.hist(zero_skill, bins=60, color="#8fa3c0", alpha=0.8)
ax.axvline(0, color="#00d4aa", lw=2, label="true Sharpe of every trader = 0")
ax.axvline(zero_skill.max(), color="#ff4757", lw=2,
           label=f"best of {N_TRADERS} = {zero_skill.max():.2f}")
ax.set_xlabel("reported 1-year Sharpe")
ax.set_ylabel("traders")
ax.set_title(f"Best-of-{N_TRADERS} among traders with no skill whatsoever")
ax.legend(frameon=False, fontsize=9)
plt.tight_layout()
plt.show()

The best of a thousand skill-free traders posts a one-year Sharpe well above 1.0, and
would pass any casual screen. Nothing about that trader is real — the number is the
maximum of a thousand draws from a distribution centred on zero.

This is the multiple-comparisons problem, and it is why "my strategy backtested at
Sharpe 2" carries almost no information unless you also know how many strategies were
tried and discarded. It applies equally to a person selecting their best year, a fund
marketing its surviving funds, and a researcher selecting the best of many
specifications.

**The honest way to report a single track record** is the point estimate with its
interval, plus the number of alternatives that were considered. This repository reports
the first two everywhere and, for the sample portfolio, the third is one — a single
fixed seed chosen before any result was seen.

---
## 4. The beta illusion

Suppose a portfolio runs leveraged market exposure through a multi-year bull run,
holding different names from the index but with no ability to pick them — its alpha is
exactly zero by construction. What does it look like on a performance report?

I simulate three years rather than one here, deliberately. Over a single year the
realised return is dominated by sampling noise and may not resemble the bull regime it
was drawn from at all; over three years it tracks the underlying drift. The longer
window also tightens the alpha confidence interval, which makes the punchline stronger
rather than weaker: even with three years of data, the estimated alpha cannot be
distinguished from zero.

In [ ]:
BULL_MU = 0.18     # benchmark drift in this scenario
BULL_VOL = 0.15
LEVERAGE = 1.8
IDIO_VOL = 0.06    # stock-specific noise, with NO expected return attached
RF = 0.02

bull_rng = np.random.default_rng(42)
BULL_YEARS = 3
n_bull = BULL_YEARS * TRADING_DAYS   # a multi-year regime, so the realised
rf_d = RF / TRADING_DAYS             # return tracks the drift instead of one noisy draw

bench_excess = bull_rng.normal(
    (BULL_MU - RF) / TRADING_DAYS, BULL_VOL / np.sqrt(TRADING_DAYS), n_bull
)
bench_ret = rf_d + bench_excess

# Zero alpha by construction: leveraged market exposure plus zero-mean noise.
# The manager picks different names from the index, but has no edge in doing so.
port_ret = (
    rf_d
    + LEVERAGE * bench_excess
    + bull_rng.normal(0.0, IDIO_VOL / np.sqrt(TRADING_DAYS), n_bull)
)

dates_bull = pd.bdate_range("2024-01-02", periods=n_bull + 1)
nav_bull = pd.Series(100.0 * np.cumprod(np.r_[1.0, 1 + port_ret]), index=dates_bull)
bench_bull = pd.DataFrame(
    {"SPY": 100.0 * np.cumprod(np.r_[1.0, 1 + bench_ret])}, index=dates_bull
)

eng_bull = MetricsEngine(nav_bull, rf=RF)
s_bull = eng_bull.summary()

print(f"Performance report over {BULL_YEARS} years for a portfolio with "
      f"ZERO alpha by construction:\n")
print(f"  Total return          {s_bull['total_return']:+.2%}")
print(f"  Annualised return     {s_bull['annualised_return']:+.2%}")
print(f"  Annualised volatility {s_bull['annualised_volatility']:.2%}")
print(f"  Sharpe ratio          {s_bull['sharpe']:.2f}  "
      f"(95% CI {s_bull['sharpe_ci_low']:.2f} to {s_bull['sharpe_ci_high']:.2f})")
print(f"  Max drawdown          {s_bull['max_drawdown']:+.2%}")
print(f"\n  Benchmark total return  {bench_bull['SPY'].iloc[-1] / 100 - 1:+.2%}")
print(f"  Portfolio outperformance "
      f"{s_bull['total_return'] - (bench_bull['SPY'].iloc[-1] / 100 - 1):+.2%}")

A cumulative return in the mid-thirties and a double-digit margin over the benchmark —
from a portfolio whose true alpha is exactly zero. This is the number that would go on
a pitch deck, and it is correctly computed. It simply answers a different question from
"is this person any good at picking stocks".

Notice what the risk column already hints at: the volatility and the drawdown are far
larger than the benchmark's, and the Sharpe ratio is unremarkable. Leverage inflates
the headline return and the risk together, so ratios expose what totals conceal. The
CAPM decomposition makes it explicit.

In [ ]:
capm_bull = BenchmarkAnalysis(eng_bull.returns, bench_bull, rf=RF).capm("SPY")

print("CAPM decomposition of the same portfolio:\n")
print(f"  beta                {capm_bull.beta:.3f}   (true leverage {LEVERAGE})")
print(f"  annualised alpha    {capm_bull.alpha_ann:+.2%}   (true alpha 0.00%)")
print(f"  alpha 95% CI        {capm_bull.alpha_ci_low:+.2%} to {capm_bull.alpha_ci_high:+.2%}")
print(f"  R-squared           {capm_bull.r_squared:.3f}")
print(f"\n  Does the alpha CI contain zero? "
      f"{capm_bull.alpha_ci_low <= 0 <= capm_bull.alpha_ci_high}")

# Attribution against the REALISED benchmark return, not the population drift.
# In arithmetic daily terms this is an exact OLS identity:
#   mean portfolio excess = alpha_daily + beta * mean benchmark excess
port_excess_ann = (eng_bull.returns.mean() - rf_d) * TRADING_DAYS
bench_excess_ann = (pd.Series(bench_ret).mean() - rf_d) * TRADING_DAYS
beta_component = capm_bull.beta * bench_excess_ann

print(f"\n  Annualised excess return (arithmetic)  {port_excess_ann:+.2%}")
print(f"  Explained by beta x market excess      {beta_component:+.2%}")
print(f"  Residual (= CAPM alpha)                "
      f"{port_excess_ann - beta_component:+.2%}")
print(f"\n  Share of excess return attributable to market beta: "
      f"{beta_component / port_excess_ann:.1%}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(nav_bull.index, nav_bull.values, color="#00d4aa", lw=2,
        label=f"Leveraged portfolio ({LEVERAGE}x, zero alpha)")
ax.plot(bench_bull.index, bench_bull["SPY"].values, color="#0099ff", lw=1.6, ls="--",
        label="Benchmark")
ax.set_ylabel("growth of 100")
ax.set_title("Impressive-looking outperformance containing no skill at all")
ax.legend(frameon=False, fontsize=9)
plt.tight_layout()
plt.show()

Beta is recovered close to the true leverage of 1.8, the estimated alpha is a rounding
error away from zero with a confidence interval comfortably spanning it, and essentially
all of the excess return is accounted for by beta multiplied by the market's own excess
return. The chart shows a portfolio handsomely beating its benchmark; the regression
shows there is no skill in it.

Note what the R² is doing here. It is high but not 1.0, because the portfolio holds
different names from the index — it has genuine idiosyncratic variation. That
variation simply has no expected return attached to it, which is precisely the
situation the CAPM regression is designed to detect: activity is not edge.

**Why this section exists.** In a bull market, leverage and skill produce the same
headline. The only way to tell them apart is to regress out the market and look at what
is left — and then to check whether what is left is larger than its own error bar. That
is the analysis I would want run on my own numbers before I quoted them to anyone, and
it is the reason this repository does not quote a personal Sharpe.

---
## 5. FIFO engine validation

The metrics above operate on a NAV series. The trade-matching engine is separate code
with separate failure modes, and it is where the original version of this project was
most wrong: it dropped short sales silently and loaded commissions without ever
subtracting them.

Each case below states the expected P&L arithmetic in the comment before the engine is
allowed to answer.

In [ ]:
def build_trades(rows):
    # rows: (date, ticker, side, qty, price, commission); side +1 buy / -1 sell.
    df = pd.DataFrame(rows, columns=["datetime", "ticker", "side", "qty", "price", "commission"])
    df["datetime"] = pd.to_datetime(df["datetime"])
    return df

def show(title, trades, expected):
    engine = FifoEngine(trades)
    realised = engine.realised_pnl()
    total = realised["net_pnl"].sum()
    print(f"{title}")
    print(f"  expected net P&L {expected:>10.2f}   engine {total:>10.2f}   "
          f"{'OK' if abs(total - expected) < 1e-9 else 'MISMATCH'}")
    assert abs(total - expected) < 1e-9, title
    return engine

In [ ]:
# Case 1 — plain long round-trip.
#   buy 10 @ 100 (comm 1.00), sell 10 @ 110 (comm 1.00)
#   gross = 10 x (110 - 100) = 100.00 ; commissions = 2.00 ; net = 98.00
show("Case 1: long round-trip", build_trades([
    ("2024-01-02", "AAA", 1, 10, 100.0, 1.0),
    ("2024-02-02", "AAA", -1, 10, 110.0, 1.0),
]), 98.00)

# Case 2 — partial close, entry commission allocated per share.
#   buy 10 @ 100 (comm 1.00 -> 0.10/share), sell 4 @ 105 (comm 0.80)
#   gross = 4 x 5 = 20.00 ; comm = 4 x 0.10 + 0.80 = 1.20 ; net = 18.80
show("Case 2: partial close", build_trades([
    ("2024-01-02", "AAA", 1, 10, 100.0, 1.0),
    ("2024-02-02", "AAA", -1, 4, 105.0, 0.8),
]), 18.80)

# Case 3 — two lots, oldest closed first.
#   buy 5 @ 100 (comm 1 -> 0.20/sh), buy 5 @ 110 (comm 1 -> 0.20/sh),
#   sell 8 @ 120 (comm 2 -> 0.25/sh)
#   lot 1: 5 x 20 = 100.00, comm 5x0.20 + 5x0.25 = 2.25, net  97.75
#   lot 2: 3 x 10 =  30.00, comm 3x0.20 + 3x0.25 = 1.35, net  28.65
#   total net = 126.40
show("Case 3: multi-lot FIFO", build_trades([
    ("2024-01-02", "AAA", 1, 5, 100.0, 1.0),
    ("2024-01-15", "AAA", 1, 5, 110.0, 1.0),
    ("2024-02-02", "AAA", -1, 8, 120.0, 2.0),
]), 126.40)

In [ ]:
# Case 4 — short sale. The old engine dropped this entirely.
#   sell 10 @ 50 (comm 1.00), buy back 10 @ 45 (comm 1.00)
#   gross = 10 x (50 - 45) = 50.00 ; comm = 2.00 ; net = 48.00
short_engine = show("Case 4: short round-trip", build_trades([
    ("2024-01-02", "SPY", -1, 10, 50.0, 1.0),
    ("2024-03-02", "SPY", 1, 10, 45.0, 1.0),
]), 48.00)
print(f"  direction recorded: {short_engine.realised_pnl().iloc[0]['direction']}")

# Case 5 — a sell with no open lot must OPEN a short, not vanish.
naked = FifoEngine(build_trades([("2024-01-02", "SPY", -1, 10, 500.0, 1.0)]))
print(f"\nCase 5: naked short opens a position")
print(f"  realised round-trips: {len(naked.realised_pnl())} (expected 0)")
print(f"  open lots: {len(naked.open_positions())} (expected 1), "
      f"direction {naked.open_positions().iloc[0]['direction']}")
assert len(naked.open_positions()) == 1

# Case 6 — flip: one sell closes a long and opens a short.
#   buy 10 @ 100 (comm 1.00), sell 15 @ 110 (comm 1.50 -> 0.10/share)
#   closes 10: gross 100.00, comm 1.00 + 10x0.10 = 2.00, net 98.00
#   opens a 5-share short @ 110 carrying 0.50 of entry commission
flip = show("\nCase 6: long-to-short flip", build_trades([
    ("2024-01-02", "CCC", 1, 10, 100.0, 1.0),
    ("2024-02-02", "CCC", -1, 15, 110.0, 1.5),
]), 98.00)
op = flip.open_positions().iloc[0]
print(f"  opened {op['direction']} {op['qty']:.0f} @ {op['entry_price']:.2f}, "
      f"entry commission {op['entry_commission']:.2f}")

Every case matches the hand arithmetic exactly. The two that the original engine got
wrong — case 4/5 (shorts) and the commission allocation in every case — are now the two
most heavily tested paths, mirrored by the `tests/test_fifo.py` suite that runs in CI.

**Why commissions matter more than they look.** On the round-trips above they shave a
percent or two off gross P&L. On a high-turnover book they are the difference between a
positive and negative strategy, and a backtest that ignores them is not a backtest of
anything tradeable. The engine allocates them per share so that a partial close carries
only its own proportional share of the opening cost.

### 5.1 Sizing: volatility targeting is not equal risk contribution

One further correctness fix worth showing. The original code described its
volatility-scaled sizer as achieving "equal risk contribution". It does not: it scales
each position by its standalone volatility and never looks at the covariance.

In [ ]:
vols = np.array([0.20, 0.20, 0.20])
corr = np.array([
    [1.00, 0.90, 0.10],
    [0.90, 1.00, 0.10],
    [0.10, 0.10, 1.00],
])
cov = np.outer(vols, vols) * corr

w_voltarget = np.full(3, 1 / 3)   # equal vols -> vol targeting gives equal weights
w_erc = erc_weights(cov)

def risk_contributions(w, cov):
    port_var = w @ cov @ w
    return w * (cov @ w) / port_var

comparison = pd.DataFrame({
    "asset": ["A (corr 0.9 with B)", "B (corr 0.9 with A)", "C (diversifier)"],
    "vol-target weight": w_voltarget,
    "vol-target risk share": risk_contributions(w_voltarget, cov),
    "ERC weight": w_erc,
    "ERC risk share": risk_contributions(w_erc, cov),
})
print("Three assets with identical volatility but very different correlations:\n")
print(comparison.round(4).to_string(index=False))

Three assets with identical volatilities get identical weights from the volatility
sizer — and wildly unequal *risk* shares, because A and B are nearly the same bet.
True ERC tilts toward the diversifier until each position contributes a third of
portfolio variance.

Both sizers are legitimate tools; they simply are not the same tool. The engine now
names them for what they are (`vol_target_size` and `erc_weights`), which is the whole
of the fix.

---
## 6. Limitations — written as the skeptical reviewer

If I were reviewing this repository and trying to find what it overstates, this is the
list I would produce. I would rather write it myself than have it produced for me.

**1. Simulated data cannot validate distributional assumptions, only arithmetic.**
Everything in sections 1–3 draws from IID normal returns. That the estimators recover
the truth *in that world* proves the formulas are implemented correctly. It says nothing
about whether real returns are IID or normal — they are neither. Section 1.3 is the
partial exception; it deliberately breaks normality to show what happens.

**2. The Lo (2002) standard error used here is the IID version.** Lo's paper also gives
a correction for autocorrelated returns, which is the case that matters for illiquid
holdings, monthly-marked positions, and anything with smoothed pricing. Autocorrelation
inflates the true standard error, so the intervals reported here are a *lower bound* on
the uncertainty for a real portfolio. Implementing the Newey-West-adjusted version is
the most valuable single extension to this code.

**3. Normality is assumed in exactly two places, and both are flagged.** Parametric VaR
and the normal-quantile Sharpe interval. The historical VaR and the bootstrap interval
are the non-parametric counterparts, which is why both are computed rather than one.

**4. CAPM is a single-factor model and it is the wrong model for most portfolios.**
A concentrated tech book loaded on momentum and growth will show "alpha" against SPY
that a Fama-French or Carhart model would reveal as factor exposure. The alpha estimates
here should be read as "return not explained by the market", not "skill".

**5. Beta is estimated over the full sample and assumed constant.** `rolling_beta`
exists and is tested, but the headline CAPM figure is a full-sample regression. A book
that changed leverage partway through would have that averaged away.

**6. The sample portfolio is not a strategy and its performance means nothing.**
It exists to exercise every code path — longs, shorts, partial fills, flips,
commissions. Its Sharpe is a property of one arbitrary random seed. Reading it as a
result would be exactly the error this notebook is about.

**7. The FIFO engine models a simplified world.** No corporate actions, no stock splits,
no dividends, no currency conversion, no margin interest, no borrow cost on shorts, and
no partial-fill slippage. On a real leveraged book, financing costs are a material drag
that this engine does not capture at all.

**8. Nothing here is out-of-sample in the machine-learning sense** because nothing here
is fitted. These are estimators with closed-form properties, not trained models. The
relevant check is calibration, which section 2 performs.

---
## 7. Conclusion — what this engine can and cannot claim

**Can claim.** The metrics are implemented correctly and demonstrably recover known
population parameters. The uncertainty quantification is calibrated: nominal 95%
intervals cover the truth close to 95% of the time in simulation. The trade-matching
engine handles long and short positions, partial fills, multi-lot FIFO ordering,
position flips, and commission allocation, verified against hand arithmetic. The
codebase is deterministic and reproducible from a fresh clone.

**Cannot claim.** Nothing about my personal trading performance. That data is private,
is not in this repository, and no figure here should be read as evidence about it.
Nothing about whether any particular strategy works. Nothing about behaviour under
non-IID or autocorrelated returns beyond what section 1.3 demonstrates.

### The quotable sentence

> Built a portfolio-analytics engine (CAPM attribution, Sharpe with Lo (2002)
> confidence intervals, parametric and historical VaR, long/short FIFO P&L) and
> validated it against simulated data with known ground truth — demonstrating via
> Monte Carlo that a one-year Sharpe estimate carries a standard error near 1.0, and
> that leveraged market beta reproduces a "skilled" performance profile with zero alpha.

Every clause maps to code in this repository:

| Claim | Where it is proved |
|---|---|
| Metrics recover known parameters | §1, `tests/test_metrics.py` |
| Sharpe CIs are calibrated | §2 coverage experiment |
| 1-year Sharpe SE ≈ 1.0 | §2, empirical SD across 2,000 simulated years |
| Track record needed for significance | §3.1 |
| Selection effect on best-of-N | §3.2 |
| Leveraged beta mimics skill | §4 |
| FIFO handles shorts and commissions | §5, `tests/test_fifo.py` |
| Vol targeting ≠ ERC | §5.1, `tests/test_sizing.py` |

### The reason the personal number is absent

The most defensible thing this repository can say about my own trading is nothing at
all. A one-year Sharpe from a personal account is a single draw from a distribution
whose standard error, as section 2 measures, is roughly the size of the estimate
itself — and it would be a number a reader could not verify even in principle.

Publishing an analysis that establishes "single-year Sharpe ratios are mostly noise" and
then quoting my own single-year Sharpe as a credential would be self-refuting. So the
claim is retired, and what remains is the part that can be checked: the engine, and the
evidence that it is correct.